# Data Collection- Steam Reviews 

## Imports 

In [1]:
import requests, time, csv, pathlib
from datetime import datetime, timezone
import pandas as pd
import numpy as np

## Saving Data

In [ ]:
DATA_PATH = pathlib.Path("data")
DATA_PATH.mkdir(exist_ok=True)


def fetch_steam_reviews(app_id: int, limit: int = 20_000, delay: float = 0.45):
    url = f"https://store.steampowered.com/appreviews/{app_id}"
    params = {
        "json": 1,
        "num_per_page": 200,
        "filter": "recent",
        "cursor": "*",
        "day_range": "30"  
    }
    rows = []
    while len(rows) < limit:
        j = requests.get(url, params=params, timeout=30).json()
        if j.get("success") != 1 or not j.get("reviews"):
            break

        for r in j["reviews"]:
            review_time = datetime.fromtimestamp(r["timestamp_created"], tz=timezone.utc)
            rows.append({
                "app_id":    app_id,
                "id":        r["recommendationid"],
                "unix_time": r["timestamp_created"],
                "thumbs_up": int(r["voted_up"]),
                "minutes_played": r["author"]["playtime_forever"],
                "useful_score": r["weighted_vote_score"],
                "text":      r["review"].replace("\r", " ").replace("\n", " ").strip(),
            })
            if len(rows) >= limit:
                break

        params["cursor"] = j["cursor"]
        time.sleep(delay)

    return rows

# Game list and scrape
APP_IDS = [730, 570, 578080]  # CS2, Dota 2, PUBG
LIMIT = 14_000

rows = []
for app in APP_IDS:
    print("Scraping", app)
    rows.extend(fetch_steam_reviews(app, LIMIT))

# Save as CSV
print("Total rows:", len(rows))
csv_path = DATA_PATH / "reviews.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

print("Saved to:", csv_path)